In [ ]:
!pip install -q "unstructured[pdf,docx]" pymupdf camelot-py[base] Pillow matplotlib
!pip install -q langchain langchain-community langchain-openai langchain-text-splitters
!pip install -q faiss-cpu pandas pydantic python-dotenv

In [3]:
import os
import io
import re
import json
import base64
import warnings
from pathlib import Path
from typing import List, Optional

import pandas as pd
import fitz  # PyMuPDF
from PIL import Image
from dotenv import load_dotenv

load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

llm        = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm_vision = ChatOpenAI(model="gpt-4o-mini", temperature=0, max_tokens=600)

SAMPLE_DIR = "samples_260522"
BATCH_DIR  = os.path.join(SAMPLE_DIR, "batch")

PDF_BUSINESS = os.path.join(SAMPLE_DIR, "business_report.pdf")
PDF_BENCH    = os.path.join(SAMPLE_DIR, "benchmark_test.pdf")
DOCX_REPORT  = os.path.join(SAMPLE_DIR, "quarterly_report.docx")


In [4]:
doc = fitz.open(PDF_BUSINESS)

In [7]:
page = doc[0]

In [8]:
len(doc)

2

In [10]:
page.rect.width, page.rect.height

(595.2755737304688, 841.8897705078125)

In [12]:
len(page.get_text('blocks'))

19

In [16]:
for b in page.get_text('blocks')[:5]:
    x0, y0, x1, y1, txt = b[0], b[1], b[2], b[3], b[4]  # x0, y0, width, height -> x0, y0, x1, y1, txt = b[0], b[1], b[0]+b[2], b[1]+b[3]
    print(f" ({x0}, {y0}) - ({x1}, {y1}) | {txt[:10]}")
    

 (56.692909240722656, 26.815746307373047) - (162.30787658691406, 35.81574630737305) | Modu Tech 
 (56.692909240722656, 792.1700439453125) - (176.26693725585938, 801.1700439453125) | © 2026 Mod
 (283.42230224609375, 817.681884765625) - (311.8533020019531, 826.681884765625) | Page 1

 (56.692909240722656, 770.6929321289062) - (242.4604949951172, 778.1929321289062) | * 본 보고서의 수
 (62.692909240722656, 80.46614837646484) - (367.19891357421875, 98.46614837646484) | Modu Tech 


In [17]:
all_tables = []
for page_idx, page in enumerate(doc):
    tables = page.find_tables()
    for t_idx, table in enumerate(tables.tables):
        df = table.to_pandas()
        all_tables.append({'page' : page_idx +1, 'shape' : df.shape, 'df' : df})

Consider using the pymupdf_layout package for a greatly improved page layout analysis.


In [18]:
len(all_tables)

2

In [19]:
for t in all_tables:
    print(f"\n Page {t['page']} | shape = {t['shape']}")
    print(t['df'].head(3).to_string())


 Page 1 | shape = (5, 4)
      사업부 매출(억) 영업이익(억) 전년동기 대비
0  AI 솔루션    98      16    +42%
1    클라우드    54       6    +18%
2     컨설팅    24       1     +5%

 Page 2 | shape = (4, 4)
     지표 2024 Q4 2025 Q4     변화율
0  고객 수   1,250   1,820  +45.6%
1   MAU     85만    132만  +55.3%
2   NPS      42      58  +38.1%


In [20]:
def analyze_tables(pdf_path) -> dict : # {'n_tables': ___ , 'tables' : [{'page', 'rows', 'cols', 'columns'}], 'total_cells'}
    doc = fitz.open(pdf_path)
    tables_info = []
    total_cells = 0
    for page_idx, page in enumerate(doc):
        tables = page.find_tables()
        for t_idx, table in enumerate(tables.tables):
            df = table.to_pandas()
            tables_info.append({'page' : page_idx +1, 'rows' : df.shape[0], 'cols' : df.shape[1], 'columns' : list(df.columns)})
            total_cells += df.shape[0] * df.shape[1]
    return {
        'n_tables' : len(tables_info),
        'tables' : tables_info,
        'total_cells' : total_cells
    }

In [22]:
analyze_tables(PDF_BUSINESS)

{'n_tables': 2,
 'tables': [{'page': 1,
   'rows': 5,
   'cols': 4,
   'columns': ['사업부', '매출(억)', '영업이익(억)', '전년동기 대비']},
  {'page': 2,
   'rows': 4,
   'cols': 4,
   'columns': ['지표', '2024 Q4', '2025 Q4', '변화율']}],
 'total_cells': 36}

In [24]:
# pymupdf (fitz) -> 의존성, 빠름, | 복잡한 표
# camelot -> 선 선명한 표에 좋다 | 라이브러리 의존성
# unstructured (hi_res) -> detectron2, yolox -> object detect



In [25]:
import camelot
cam_tables = camelot.read_pdf(PDF_BUSINESS, pages= 'all', flavor='lattice')
cam_tables

<TableList n=3>

In [27]:
for i, t in enumerate(cam_tables):
    print(f"Table {i+1} | page = {t.page} | shape = {t.df.shape} | accuracy = {t.accuracy}")
    print(t.df.head(2).to_string())

Table 1 | page = 1 | shape = (5, 10) | accuracy = 0
  0 1 2 3 4 5 6 7 8 9
0                    
1                    
Table 2 | page = 1 | shape = (6, 4) | accuracy = 100.00000000000003
        0      1        2        3
0     사업부  매출(억)  영업이익(억)  전년동기 대비
1  AI 솔루션     98       16     +42%
Table 3 | page = 2 | shape = (5, 4) | accuracy = 100.0
      0        1        2       3
0    지표  2024 Q4  2025 Q4     변화율
1  고객 수    1,250    1,820  +45.6%


In [30]:
# NA - 

In [29]:
from unstructured.partition.pdf import partition_pdf
els = partition_pdf(filename = PDF_BUSINESS, strategy = 'hi_res')
unst_tables = [el for el in els if el.category == 'Table']
for tb in unst_tables:
    print(f"\n page = {tb.metadata.page_number}")
    print(str(tb)[:50])

/home/oncreative/anaconda3/envs/modu/lib/python3.11/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
No languages specified, defaulting to English.



 page = 1
사업부 매출(억) 영업이익(억) 전년동기 대비 AI 솔루션 98 16 +42% 클라우드 5

 page = 2
지표 2024 Q4 2025 Q4 변화율 고객 수 1,250 1,820 +45.6% MAU


In [31]:
doc = fitz.open(PDF_BUSINESS)
raw_df = doc[0].find_tables().tables[0].to_pandas()

In [32]:
raw_df

,사업부,매출(억),영업이익(억),전년동기 대비
0,AI 솔루션,98,16,+42%
1,클라우드,54,6,+18%
2,컨설팅,24,1,+5%
3,기타,12,1,-2%
4,합계,188,24,+35%


In [34]:
print(raw_df.to_string())

      사업부 매출(억) 영업이익(억) 전년동기 대비
0  AI 솔루션    98      16    +42%
1    클라우드    54       6    +18%
2     컨설팅    24       1     +5%
3      기타    12       1     -2%
4      합계   188      24    +35%


In [ ]:
# 1) 첫번째 행이 헤더로 보이면 -> 컬럼명으로 올려주기
# 2) NA, -, "" -> 통일
# 3) 숫자처럼 보이는 열은 numeric

In [36]:
def clean_table(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
#     if df.iloc[0].apply(lambda v:isinstance(v, str) and len(v) < 30).all():
#         df.columns = df.iloc[0]
#         df = df.iloc[1:].reset_index(drop=True)
    
    df = df.replace({'N/A' : None, '-' : None, '' : None})
    
    for col in df.columns:
        sample = df[col].dropna().astype(str)
        if sample.empty:
            continue
        if sample.str.match(r"^[\$\+\-]?[\d,]+\.?\d*%?$").mean() > 0.6:
            cleaned = df[col].astype(str).str.replace(r"[\$,%]", "", regex=True)
            df[col] = pd.to_numeric(cleaned)
    return df

In [37]:
clean_df = clean_table(raw_df)

In [39]:
# clean_df

In [41]:
def table_to_markdown(df : pd.DataFrame, max_rows : int = 10) -> str:
    df = df.fillna("")
    header = "| "+" | ".join(str(c) for c in df.columns) + " |"
    sep = "| " + " | ".join(["---"] * len(df.columns)) + " |"
    rows = df.head(max_rows).astype(str).apply(
            lambda r : "| " + " | ".join(r) + " |", axis=1).tolist()
    out = '\n'.join([header, sep] + rows)
    
    if len(df) > max_rows:
        out += f"\n ... ({len(df) - max_rows} more rows)"
    return out

In [42]:
md_text = table_to_markdown(clean_df)
print(md_text)

| AI 솔루션 | 98 | 16 | +42% |
| --- | --- | --- | --- |
| 클라우드 | 54 | 6 | 18 |
| 컨설팅 | 24 | 1 | 5 |
| 기타 | 12 | 1 | -2 |
| 합계 | 188 | 24 | 35 |


In [ ]:
doc = fitz.open(PDF_BUSINESS)
saved = []
for page_idx, page in enumerate(doc):
    for img_ix, img in enumerate(page.get_images(full=True)):
        xref = img[0]
        base_img = doc.extract_image(xref)
        ext = base_img['ext']
        path = os.path.join(f"'./260522_images', 'abc.{ext}'")
        with open(path, 'wb') as f:
            f.write(base_img['image'])
            

In [45]:
def analyze_image(image_path, prompt='이 이미지에 보이는 모든 텍스트를 추출하고 한 문장으로 요약해줘'):
    with open(image_path, 'rb') as f:
        b64 = base64.b64encode(f.read()).decode('utf-8')
    ext = os.path.splitext(image_path)[1].lstrip('.').lower()
    if ext == 'jpg':
        ext = 'jpeg'
    
    msg = HumanMessage(content = [
        {'type' : 'text', 'text' : prompt},
        {'type' : 'image_url', 'image_url' : {'url' : f'data:image/{ext}:base64,{b64}'}}
    ])
    return llm_vision.invoke([msg]).content

In [46]:
img_dir = './260522_images'
imgs = ['./260522_images/p1_img1.png', './260522_images/p2_img1.png']

In [47]:
out = analyze_image(imgs[0])

In [49]:
print(out)

**추출된 텍스트:**  
"2025 Quarterly Revenue (억 원)"  
"Q1 120"  
"Q2 145"  
"Q3 162"  
"Q4 188"  

**요약:**  
2025년 분기별 수익은 Q1 120억 원, Q2 145억 원, Q3 162억 원, Q4 188억 원이다.


In [50]:
from langchain_core.documents import Document

class VisionOCR:
    def __init__(self, llm):
        self.llm = llm
        self.results = []
    
    def process(self, paths: list, prompt : str = "이미지에서 모든 내용을 추출하세요") -> list:
        self.results = []
        for p in paths:
            try:
                with open(p, 'rb') as f:
                    b64 = base64.b64encode(f.read()).decode('utf-8')
                ext = os.path.splitext(p)[1].lstrip('.').lower()
                if ext == 'jpg':
                    ext = 'jpeg'

                msg = HumanMessage(content = [
                    {'type' : 'text', 'text' : prompt},
                    {'type' : 'image_url', 'image_url' : {'url' : f'data:image/{ext}:base64,{b64}'}}
                ])
                self.results.append({'path' :p, 'text' : self.llm.invoke([msg]).content, 'error' : None})
            except Exception as e:
                self.results.append({'path' : p, 'text' : '', 'error' : str(e)})
    
    def to_documents(self) -> list:
        return [Document(page_content = r['text'], metadata = {'path' : r['path']}) for r in self.results if r['text']]

In [51]:
ocr = VisionOCR(llm_vision)

In [52]:
res = ocr.process(imgs[:2])

In [53]:
ocr.results

[{'path': './260522_images/p1_img1.png',
  'text': '이미지에서 추출한 내용은 다음과 같습니다:\n\n- 제목: 2025 Quarterly Revenue (억 원)\n- Q1: 120억 원\n- Q2: 145억 원\n- Q3: 162억 원\n- Q4: 188억 원\n\nY축의 범위는 0에서 226까지입니다.',
  'error': None},
 {'path': './260522_images/p2_img1.png',
  'text': '이미지에서 추출한 내용은 다음과 같습니다:\n\n- 제목: "월별 매출 추이 vs 시장 평균"\n- x축: "Jan", "Feb", "Mar", "Apr", "May", "Jun"\n- y축: 값의 범위는 0.0에서 178.2까지\n- 두 개의 선 그래프:\n  - 하나는 초록색으로 표시된 매출 추이\n  - 다른 하나는 빨간색으로 표시된 시장 평균\n\n각각의 데이터 포인트는 다음과 같습니다:\n- 매출 추이: 89.1 (Jan), 133.7 (Jun)\n- 시장 평균: 44.6 (Jan), 89.1 (Jun) \n\n이 그래프는 두 가지 추세를 비교하고 있습니다.',
  'error': None}]

In [54]:
docs = ocr.to_documents()

In [55]:
docs

[Document(metadata={'path': './260522_images/p1_img1.png'}, page_content='이미지에서 추출한 내용은 다음과 같습니다:\n\n- 제목: 2025 Quarterly Revenue (억 원)\n- Q1: 120억 원\n- Q2: 145억 원\n- Q3: 162억 원\n- Q4: 188억 원\n\nY축의 범위는 0에서 226까지입니다.'),
 Document(metadata={'path': './260522_images/p2_img1.png'}, page_content='이미지에서 추출한 내용은 다음과 같습니다:\n\n- 제목: "월별 매출 추이 vs 시장 평균"\n- x축: "Jan", "Feb", "Mar", "Apr", "May", "Jun"\n- y축: 값의 범위는 0.0에서 178.2까지\n- 두 개의 선 그래프:\n  - 하나는 초록색으로 표시된 매출 추이\n  - 다른 하나는 빨간색으로 표시된 시장 평균\n\n각각의 데이터 포인트는 다음과 같습니다:\n- 매출 추이: 89.1 (Jan), 133.7 (Jun)\n- 시장 평균: 44.6 (Jan), 89.1 (Jun) \n\n이 그래프는 두 가지 추세를 비교하고 있습니다.')]